# 2: Information Theory

*The vocabulary the whole project's math is written in.*

This notebook introduces the core information-theoretic ideas behind uncertainty, prediction quality, and model calibration in LLMs. The goal is not to drown in formulas, but to build the intuition that will keep showing up later in entropy-based methods, confidence estimation, and semantic clustering.

We will cover:

- 02a. Shannon entropy
- 02b. Cross-entropy and KL divergence
- 02c. Perplexity
- 02d. Mutual information

In [7]:
import numpy as np
import math
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True)

# Small helper function for entropy

def entropy_from_probs(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

# Small helper function for cross-entropy

def cross_entropy_from_probs(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return -(p * np.log2(q)).sum()

# Small helper function for KL divergence

def kl_divergence(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return (p * np.log2(p / q)).sum()

## 2a. Shannon entropy

Entropy measures the uncertainty or surprise in a random variable. It answers the question:

> If I observe a sample from this distribution, how much information do I expect to gain?

For a discrete random variable $X$ with outcomes $x_1, \dots, x_n$ and probabilities $p_1, \dots, p_n$:

$$
H(X) = -\sum_i p_i \log_2 p_i
$$

The units are bits because we use $\log_2$.

### Intuition

- A very predictable event has low entropy.
- A highly uncertain event has high entropy.
- A fair coin has maximum uncertainty among two outcomes:
  $$H(X) = 1 \text{ bit}$$
- A deterministic event has entropy 0.

### Worked by hand: fair coin

If $P(H)=0.5$ and $P(T)=0.5$:

$$
H(X) = -(0.5\log_2 0.5 + 0.5\log_2 0.5)
= -2(0.5 \cdot (-1)) = 1
$$

So one fair coin flip carries 1 bit of information.

### Worked by hand: biased coin

Suppose $P(H)=0.8$ and $P(T)=0.2$:

$$
H(X) = -(0.8\log_2 0.8 + 0.2\log_2 0.2)
\approx 0.722 \text{ bits}
$$

This is lower than 1 bit because the event is more predictable.

### Worked by hand: 6-sided die

For a fair die:

$$
H(X) = -\sum_{i=1}^{6} \frac{1}{6}\log_2\left(\frac{1}{6}\right)
= \log_2 6 \approx 2.585 \text{ bits}
$$

This is the maximum uncertainty for a 6-way choice.

### Why this matters for LLMs

A language model assigns probabilities to possible next tokens. The entropy of that distribution tells us how uncertain the model is about the next word.

- High entropy: many plausible next tokens
- Low entropy: the model is confident

This is often used as a rough uncertainty signal.

In [8]:
# Entropy examples for simple distributions

examples = {
    "fair_coin": [0.5, 0.5],
    "biased_coin": [0.8, 0.2],
    "fair_die": [1/6] * 6,
    "very_confident": [0.95, 0.05],
}

for name, probs in examples.items():
    h = entropy_from_probs(probs)
    print(f"{name}: entropy = {h:.4f} bits")

# A model output distribution over 4 next tokens
model_probs = np.array([0.5, 0.25, 0.15, 0.10])
print(f"Model entropy = {entropy_from_probs(model_probs):.4f} bits")

fair_coin: entropy = 1.0000 bits
biased_coin: entropy = 0.7219 bits
fair_die: entropy = 2.5850 bits
very_confident: entropy = 0.2864 bits
Model entropy = 1.7427 bits


### A quick sanity check

Entropy is highest for a uniform distribution over possible outcomes.

For a finite alphabet of size $n$, the maximum possible entropy is:

$$
H_{max} = \log_2 n
$$

So a fair coin gives $\log_2 2 = 1$, and a fair die gives $\log_2 6 \approx 2.585$.

This is one reason entropy is such a useful summary: it compresses a whole distribution into a single uncertainty number.

## 2b. Cross-entropy & KL divergence

Now we ask a different question:

> If the true distribution is $p$, but I use model distribution $q$, how much penalty do I incur?

That is exactly the idea behind cross-entropy.

### Cross-entropy

$$
H(p, q) = -\sum_i p_i \log_2 q_i
$$

This is the expected number of bits required to encode samples from the true distribution $p$ when the code is optimized for $q$ instead.

### Relationship to training loss

In language modeling, the training objective is usually the negative log-likelihood (NLL), or equivalently cross-entropy.

If the true next-token distribution is $p$ and the model predicts $q$, then:

$$
\text{Loss} = -\log q(y)
$$

and across many examples, the average loss is close to cross-entropy.

This is why cross-entropy is not just a theoretical concept; it is the practical loss the model is trained to minimize.

### KL divergence

KL divergence measures how different two distributions are:

$$
D_{KL}(p \| q) = \sum_i p_i \log_2\left(\frac{p_i}{q_i}\right)
$$

It is connected to cross-entropy by:

$$
H(p, q) = H(p) + D_{KL}(p \| q)
$$

So cross-entropy decomposes into:

1. the intrinsic uncertainty of the true distribution $H(p)$
2. the extra penalty from using the wrong distribution $q$

When the model is perfect, $q = p$, and then:

$$
H(p, q) = H(p), \quad D_{KL}(p \| q) = 0
$$

### Intuition

- Entropy measures uncertainty in one distribution.
- Cross-entropy measures the cost of using one distribution to describe another.
- KL divergence measures how far apart the distributions are.

### Tiny worked example

True distribution:

$$
 p = [0.8, 0.2]
$$

Model prediction:

$$
 q = [0.7, 0.3]
$$

Then:

$$
H(p) = -(0.8\log_2 0.8 + 0.2\log_2 0.2) \approx 0.722
$$

and

$$
H(p, q) = -(0.8\log_2 0.7 + 0.2\log_2 0.3) \approx 0.884
$$

so

$$
D_{KL}(p \| q) = H(p, q) - H(p) \approx 0.162
$$

This tells us the model is slightly mis-specified, and the extra penalty is about 0.162 bits.

In [9]:
# Cross-entropy and KL example

p = np.array([0.8, 0.2])
q = np.array([0.7, 0.3])

H_p = entropy_from_probs(p)
H_pq = cross_entropy_from_probs(p, q)
kl = kl_divergence(p, q)

print(f"True entropy H(p) = {H_p:.4f} bits")
print(f"Cross-entropy H(p, q) = {H_pq:.4f} bits")
print(f"KL divergence D_KL(p || q) = {kl:.4f} bits")
print(f"Check: H(p, q) - H(p) = {H_pq - H_p:.4f}")

# Compare with a perfect model
q_perfect = p
print(f"Perfect model cross-entropy = {cross_entropy_from_probs(p, q_perfect):.4f} bits")

True entropy H(p) = 0.7219 bits
Cross-entropy H(p, q) = 0.7591 bits
KL divergence D_KL(p || q) = 0.0371 bits
Check: H(p, q) - H(p) = 0.0371
Perfect model cross-entropy = 0.7219 bits


## 2c. Perplexity

Perplexity is the most familiar information-theoretic quantity in NLP.

It is defined as:

$$
\text{Perplexity} = 2^{H(p, q)}
$$

or, equivalently,

$$
\text{Perplexity} = \exp(H(p, q) \cdot \ln 2)
$$

When using the model's predictive distribution $q$ on a true target distribution $p$, perplexity is just the exponential of cross-entropy.

### Why it is used in NLP

For language models, lower perplexity means the model is better at predicting the next token. A model with perplexity 10 is, in a rough sense, as uncertain as a uniform distribution over 10 equally plausible next tokens.

### Important connection

Perplexity is not a different concept from cross-entropy. It is a rescaled version of it.

- lower cross-entropy = better prediction quality
- lower perplexity = same thing, just in a different scale

### Example

If cross-entropy is 2.5 bits, then:

$$
\text{Perplexity} = 2^{2.5} \approx 5.66
$$

So the model is roughly as uncertain as choosing among about 6 equiprobable next tokens.

In [10]:
# Perplexity example

def perplexity_from_cross_entropy(ce):
    return 2 ** ce

ce_values = [1.0, 2.0, 3.0, 4.0]
for ce in ce_values:
    pp = perplexity_from_cross_entropy(ce)
    print(f"Cross-entropy = {ce:.1f} bits -> Perplexity = {pp:.3f}")

# Example with a token distribution
p = np.array([0.8, 0.2])
q = np.array([0.7, 0.3])
ce = cross_entropy_from_probs(p, q)
pp = perplexity_from_cross_entropy(ce)
print(f"Example CE = {ce:.4f}, Example PPL = {pp:.4f}")

Cross-entropy = 1.0 bits -> Perplexity = 2.000
Cross-entropy = 2.0 bits -> Perplexity = 4.000
Cross-entropy = 3.0 bits -> Perplexity = 8.000
Cross-entropy = 4.0 bits -> Perplexity = 16.000
Example CE = 0.7591, Example PPL = 1.6924


## 2d. Mutual information

This is a stretch topic, but it is worth knowing because it appears later in discussions of semantic uncertainty and clustering.

Mutual information measures how much knowing one variable tells us about another.

$$
I(X; Y) = H(X) - H(X \mid Y)
$$

Equivalently,

$$
I(X;Y) = D_{KL}(p(x,y) \| p(x)p(y))
$$

### Intuition

Mutual information tells us whether two variables contain shared information.

- If $I(X;Y)=0$, then $X$ and $Y$ are independent.
- If $I(X;Y)$ is large, then learning $Y$ gives strong information about $X$.

### Why it matters here

In semantic uncertainty methods, we often ask whether multiple model hypotheses or sampled generations share latent structure. Mutual information is a compact way to quantify whether one source of information reduces uncertainty in another.

For this project, it is mostly a supporting concept. We do not need a heavy treatment now; just enough to understand that information is not only about entropy in one distribution, but also about dependency between variables.

### Very brief example

Suppose $X$ is a random variable representing whether a fact is true, and $Y$ is a model's confidence signal. If $Y$ strongly correlates with correctness, then $I(X;Y)$ is nonzero and meaningful.

If $Y$ is uninformative, then $I(X;Y)$ is close to zero.

## Summary

These four ideas form the basic vocabulary of information theory:

- entropy measures uncertainty in a single distribution
- cross-entropy measures prediction cost under a mismatched model
- KL divergence measures how far two distributions are apart
- perplexity is just cross-entropy on a more familiar scale
- mutual information measures how much two variables tell us about each other

For LLM work, this is exactly the language behind:

- next-token uncertainty
- negative log-likelihood training loss
- calibration and confidence estimation
- semantic entropy and distributional comparisons

The next time you see a model “confidently wrong,” the information-theoretic view explains why: the model is assigning a sharp distribution to the wrong event, which raises cross-entropy and lowers predictive quality even when entropy seems low.